<a href="https://colab.research.google.com/github/shibataryoma22-hub/-udemy/blob/main/mnist_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transform
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
#訓練データ
train_dataset = torchvision.datasets.MNIST(root='./data',
                                           train=True,
                                           transform=transform.ToTensor(),
                                           download = True)
#検証データ
test_dataset = torchvision.datasets.MNIST(root='./data',
                                           train=False,
                                           transform=transform.ToTensor(),
                                           download = True)
print("train_dataset\n", train_dataset)
print("test_dataset\n", test_dataset)

In [ ]:
#データセット、データローダの作成
batch_size = 256
train_loader = torch.utils.data.DataLoader(dataset = train_dataset,
                                           batch_size = batch_size,
                                           shuffle = True)

test_loader = torch.utils.data.DataLoader(dataset = test_dataset,
                                          batch_size = batch_size,
                                          shuffle = True
                                          )

In [ ]:
#モデルの作成
class Net(nn.Module):
    def __init__(self, input_size, hidden1_size, hidden2_size, output_size):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden1_size)
        self.fc2 = nn.Linear(hidden1_size, hidden2_size)
        self.fc3 = nn.Linear(hidden2_size,output_size)

    #順伝搬の定義
    def forward(self, x): #xは入力
        z1 = F.relu(self.fc1(x))
        z2 = F.relu(self.fc2(z1))
        y = self.fc3(z2) #本来活性化関数のsoftmaxが来るはずであるが、
                         #PyTorchのnn.CrossEntropyLoss()にはsoftmax()
                         #が含まれているため順伝搬部分に書く必要はない
        return y

In [ ]:
#モデルのインスタンス化をする
input_size = 28 * 28 #入力画像のサイズ
hidden1_size = 1024 #任意
hidden2_size = 512 #任意
output_size = 10 #クラス数

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print("device", device)

model = Net(input_size, hidden1_size, hidden2_size, output_size).to(device)
print(model)

In [ ]:
criterion = nn.CrossEntropyLoss() #交差エントロピー誤差
optimizer = optim.SGD(model.parameters(), lr = 0.01) #lr:学習率

データを1バッチ取り出す
        ↓
順伝搬（予測）
        ↓
損失を計算
        ↓
誤差逆伝搬
        ↓
重みを更新
        ↓
次のバッチへ

In [ ]:
def train_model(model, train_loader, criterion, optimizer):
    train_loss = 0.0 #trainの損失用の変数を定義
    num_train = 0 #学習回数の記録用の変数を定義

    #モデルを学習モードに変換する
    model.train()

    #データを分割数分繰り返す
    #バッチサイズ分のデータで１回パラメータを修正する
    for i, (images, labels) in enumerate(train_loader):

        #batch数をカウント
        num_train += len(labels)
        images, labels = images.view(-1, 28*28).to(device), labels.to(device)
        #勾配の初期化
        optimizer.zero_grad()
        #順伝搬
        outputs = model(images)
        #損失の算出
        loss = criterion(outputs, labels)
        #勾配計算
        loss.backward()
        #パラメータの更新
        optimizer.step()
        #lossを加算
        train_loss += loss.item()

    #lossの平均値をとる
    train_loss = train_loss / num_train

    return train_loss

In [ ]:
def test_model(model, test_loader, criterion, optimizer):

    test_loss = 0.0
    num_test = 0

#モデルを評価用に変更
    model.eval()
    with torch.no_grad():
        for i, (images, labels) in enumerate(test_loader):
            num_test += len(labels)
            images, labels = images.view(-1, 28*28).to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()

        #lossの平均値をとる
        test_loss = test_loss / num_test
    return test_loss


In [ ]:
#モデルの学習を行う関数の定義
def learning(model, train_loader, test_loader, criterion, optimizer, num_epochs):
    train_loss_list = []
    test_loss_list = []
    #epoch数分繰り返す
    for epoch in range(1, num_epochs+1, 1):
        train_loss = train_model(model, train_loader, criterion, optimizer)
        test_loss = test_model(model, test_loader, criterion, optimizer)

        print("epoch:{}, train_loss:{:.5f}, test_loss:{:.5f}".format(epoch, train_loss, test_loss))

        train_loss_list.append(train_loss)
        test_loss_list.append(test_loss)
    return train_loss_list, test_loss_list


In [ ]:
#学習を行う
num_epochs = 10
train_loss_list, test_loss_list = learning(model, train_loader, test_loader, criterion, optimizer, num_epochs)
